# 
Say what it is about in general

In [5]:
# imports

# you can use this for averaging etc if you want
import numpy

# import whatever you want from your own code, for instance:
from corpusreader import CorpusReader
from generate import *
from model import BigramModel

## Build the model
We begin by building our model from the training set. We use this throughout

In [12]:
# set path to training set
path =r"train"


In [13]:
# build the model
my_model = BigramModel(CorpusReader(path).sents())

## Experimenting with smoothing constants

Smoothing constants are used both for generating samples and for computing the perplexity of text. To investigate the effect of smoothing constants on generating samples and computing perplexity of samples, we provide functions that allow us to try a range of smoothing constants.

### Generating a sample with a particular smoothing constant

We made two functions; generate_sample that gives us a sample corpus generated by the model, and compute_corpus_perplexity to calculate its perplexity. Both with possibly different smoothing constants.

In [14]:
def generate_sample(model, sample_size, smoothing_constant):
    """
    Generate a sample of size sample_size from model, using smoothing_constant for LaPlace smoothing
    @param model: BigramModel
    @param sample_size: int
    @param smoothing_constant: float 
    @return: list of lists of strings
    """

    lijst = []
    for i in range(0, sample_size):
      lijst.append(generate_sentence(model, smoothing_constant)[1:-1])

    print("\ngenerating with constant", smoothing_constant)
    return lijst


In [15]:
def compute_corpus_perplexity(model, sample, smoothing_constant):
    """
    returns the average perplexity of the given sample with the model, using the smoothing constant for add-k smoothing
    @param model: ...
    @param sample: ...
    @param smoothing_constant: ...
    @return: ...
    """
    average = []
    for i in range(0, len(sample)):
      average.append(model.perplexity(sample[i],smoothing_constant))
    return sum(average)/len(average)


To try this out, we generate a sample of 10 sentences using raw probabilities (k = 0.0) and test it with LaPlace smoothing (k = 1.0)

In [20]:
n = 5
raw_sample = generate_sample(my_model, n, 0.0)


generating with constant 0.0


We can see some examples below

In [24]:
print("5 examples:\n")
for s in raw_sample[:5]:
    print(" ".join(s))

5 examples:

wrong
were for that this time has captured coach was hardly maintain i concede the omnibus
capable of a bison
at the newspapers
he preferred a man


To calculate the perplexity of this sample, we can choose a smoothing constant -- let's make it 1.0

In [25]:
laplace_constant = 1.0
print(compute_corpus_perplexity(my_model, raw_sample, laplace_constant))

1.0000995122250487


To see how the perplexity calculation of this sample varies when we change the smoothing constant we use to calculate it, we can test with 10 fractional constants

In [38]:
for i in range(1, 11):  # should this be 0 or 1?
    c = i/10  # 0.0, 0.1, 0.2, ..., 1.0
    print("\nTest with constant", c)
    print(compute_corpus_perplexity(my_model, raw_sample, c))


Test with constant 0.1
1.0000975407204673

Test with constant 0.2
1.0000978057974286

Test with constant 0.3
1.0000980558993189

Test with constant 0.4
1.0000982929580755

Test with constant 0.5
1.0000985185320803

Test with constant 0.6
1.0000987338991094

Test with constant 0.7
1.000098940121679

Test with constant 0.8
1.000099138094153

Test with constant 0.9
1.0000993285774398

Test with constant 1.0
1.0000995122250487


### Analysis

We can see that as we increase the smoothing constant used to calculate the perplexity, the perplexity also increases. This is actually counterintuitive as it is usually expected that the perplexity decreases when the smoothing constant used to calculate it is increased. This is because the add-k smoothing technique we used adds probability mass to unseen events, therefore decreasing the penalty for rare events.

## Varying the smoothing constant used for generation

Above we used raw probabilities. Now we will see what happens when we vary the constant used to generate the sample

In [40]:
for i in range(1, 11):  # should this be 0 or 1?
    n = 5
    c = i/10
    sample = generate_sample(my_model, n, c)
    print(compute_corpus_perplexity(my_model, sample, 0.0))


generating with constant 0.1
1.000222300430925

generating with constant 0.2
1.0001425335281104

generating with constant 0.3
1.0002847861619983

generating with constant 0.4
1.0001650963403905

generating with constant 0.5
1.0002664262678642

generating with constant 0.6
1.0002401192083765

generating with constant 0.7
1.000466669602088

generating with constant 0.8
1.000156081897668

generating with constant 0.9
1.0001439991167707

generating with constant 1.0
1.000226611677694


### Analysis

It seems that the perplexity increases together with the smoothing constant used to generate the samples, but when taking a closer look we see that this increase is not as stable as the previous test where we increased the calculating smoothing constant. There is a lot more variety in the results and this can be explained by the result smoothing has on the generation of samples. A higher smoothing constant increases the probability of rare or unseen events, therefore increasing the overall randomness of the sample. Because of this, the new sample can deviate more from the original corpus, resulting in a higher complexity, but because this is random, we don't see a steady increase here.

It could be interesting to see more combinations of smoothing constants (both for generating and calculating) but we expect that the further increase in randomness will make the conclusions we draw from it unsignifficant. 

## Discussion

As said above, the results were unexpected. After reading about the effects of different smoothing constants we expected to see a decrease of the perplexity when increasing the smoothing constant used for calculating. In our experiment this was not the case. This could be caused by a number of factors. One possible reason for this increase in perplexity could be that add-k smoothing tends to overestimate the probability of unseen events, thereby distorting the results too much. We also found that in cases where the training data is relatively large or the corpus contains a wide variety of events, add-k smoothing may distribute the additional probability mass too evenly, resulting in an increase in perplexity.
In contrast, the results for increasing the smoothing constant for generating samples did give us the expected results. We can see that the increased constant causes the samples to be more randomly generated.

## Conclusion

We found that changing the value of the smoothing constants has an effect on the overall perplexity. While increasing the smoothing constant used to calculate the perplexity, the perplexity increases. And while increasing the smoothing constant used to generate the samples, the perplexity decreases. 
Both constants interact with each other. For instance, if the smoothing constant used to generate text is significantly higher - increasing diversity - but the perplexity is calculated with a lower smoothing constant that emphasises the original distribution, the perplexity value might not be very accurate in reflecting the divergence between the generated text and the original corpus.